# 04 — Train the CNN
Build and train a 1D CNN that classifies radar signal types directly from raw IQ data.

In [ ]:
import os
import random
import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib.pyplot as plt

DATA_PATH = '../data/RadChar-Small.h5'

# One seed, set across EVERY source of randomness (python, numpy, torch).
# Using the same seed everywhere is what makes the split and training
# reproducible — and lets notebook 05 evaluate on the exact same test set
# the model never trained on (no data leakage).
SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)

# use GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
# load raw IQ and signal type labels
with h5py.File(DATA_PATH, 'r') as f:
    iq     = f['iq'][:]
    labels = f['labels'][:]

In [ ]:
# split complex IQ into two separate channels: I (real) and Q (imaginary)
# shape goes from (N, 512) complex -> (N, 2, 512) float
I = iq.real.astype(np.float32)
Q = iq.imag.astype(np.float32)
X = np.stack([I, Q], axis=1)  # (N, 2, 512)

# signal type is what we want to predict (0, 1, 2, 3, 4)
y = labels['signal_type'].astype(np.int64)

# free the large complex array — we only need X from here on.
# matters on the Small set (iq alone is ~4 GB in memory).
del iq, I, Q

print('Input shape:', X.shape)
print('Labels shape:', y.shape)

In [ ]:
# PyTorch dataset — wraps X and y so DataLoader can batch them
class RadCharDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


dataset = RadCharDataset(X, y)

# --- deterministic 70/15/15 split ---
# Best practice: split BEFORE any fitting/preprocessing, and never let the test
# set influence training. We shuffle indices once with the fixed seed, slice, and
# SAVE the indices so notebook 05 reuses the identical split. The CNN trains on
# raw I/Q with no normalisation, so there is nothing fitted that could leak.
N = len(dataset)
perm = np.random.permutation(N)

n_train = int(0.70 * N)
n_val   = int(0.15 * N)

train_idx = perm[:n_train]
val_idx   = perm[n_train:n_train + n_val]
test_idx  = perm[n_train + n_val:]

os.makedirs('../results', exist_ok=True)
np.save('../results/train_idx.npy', train_idx)
np.save('../results/test_idx.npy',  test_idx)

train_set = Subset(dataset, train_idx)
val_set   = Subset(dataset, val_idx)
test_set  = Subset(dataset, test_idx)

# seed the shuffling generator too, so batch order is reproducible
g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, generator=g)
val_loader   = DataLoader(val_set,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=64, shuffle=False)

print(f'Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}')
print('Saved split indices to results/train_idx.npy and results/test_idx.npy')

In [ ]:
# 1D CNN model
# input:  (batch, 2, 512)  — 2 channels, 512 time steps
# output: (batch, 5)       — score for each signal type
class RadarCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            # block 1 — find low-level patterns
            nn.Conv1d(2,  32, kernel_size=11, padding=5), nn.ReLU(), nn.MaxPool1d(2),
            # block 2 — find higher-level patterns
            nn.Conv1d(32, 64, kernel_size=7,  padding=3), nn.ReLU(), nn.MaxPool1d(2),
            # block 3 — find even higher-level patterns
            nn.Conv1d(64, 128, kernel_size=5, padding=2), nn.ReLU(),
            # collapse time dimension to a fixed size regardless of input length
            nn.AdaptiveAvgPool1d(1),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Dropout(0.5),     # increased from 0.3 to reduce overfitting
            nn.Linear(64, 5),    # 5 output classes
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = RadarCNN().to(device)
print(model)

In [ ]:
# loss function and optimiser
# weight_decay adds a small penalty for large weights — this is L2 regularisation
criterion = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
# training loop with early stopping
# early stopping: if val loss doesn't improve for PATIENCE epochs, stop early
# this prevents the model from training past the point where it starts to overfit
EPOCHS   = 25
PATIENCE = 5   # stop if no improvement for 5 epochs in a row

train_losses, val_losses, val_accs = [], [], []

best_val_loss     = float('inf')
patience_counter  = 0
best_model_state  = None

for epoch in range(EPOCHS):

    # --- train ---
    model.train()
    running_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimiser.zero_grad()
        preds = model(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        optimiser.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    # --- validate ---
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds  = model(X_batch)
            loss   = criterion(preds, y_batch)
            val_loss  += loss.item()
            correct   += (preds.argmax(1) == y_batch).sum().item()
            total     += y_batch.size(0)

    val_loss = val_loss / len(val_loader)
    val_acc  = correct / total

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    # --- early stopping check ---
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        best_model_state = model.state_dict().copy()   # snapshot the best weights
        tag = ' ← best'
    else:
        patience_counter += 1
        tag = f' (no improvement {patience_counter}/{PATIENCE})'

    print(f'Epoch {epoch+1:02d}/{EPOCHS} | train loss: {train_loss:.4f} | val loss: {val_loss:.4f} | val acc: {val_acc:.3f}{tag}')

    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1} — val loss stopped improving.')
        break

# restore the best weights (not the last ones)
model.load_state_dict(best_model_state)
print(f'\nRestored best model (val loss: {best_val_loss:.4f})')

In [ ]:
# save the trained model weights so notebook 05 can load them
torch.save(model.state_dict(), '../results/radar_cnn.pth')
print('Model saved to results/radar_cnn.pth')

In [ ]:
# plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, label='train loss')
axes[0].plot(val_losses,   label='val loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(val_accs, color='green', label='val accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# final evaluation on the test set — run this only once after training is done
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        preds    = model(X_batch)
        correct += (preds.argmax(1) == y_batch).sum().item()
        total   += y_batch.size(0)

test_acc = correct / total
print(f'Test accuracy: {test_acc:.3f} ({test_acc*100:.1f}%)')